# Practice Session 05: Hubs and authorities


<font size="+2" color="blue">Additional results: country clusters</font>

# 1. Read mapping of codes to country names

In [ ]:
import csv
import io
import networkx as nx
import matplotlib.pyplot as plt
import math
import pandas as pd

In [ ]:
# Leave this cell as-is

INPUT_NAMES_FILENAME = "trade-countries.csv"
INPUT_TRADE_OLD = "trade-2010-flows.csv"
YEAR_OLD = 2010
INPUT_TRADE_NEW = "trade-2021-flows.csv"
YEAR_NEW = 2021

In [ ]:
id2name = {}

# Open a file for reading in text mode
with open(INPUT_NAMES_FILENAME, "rt") as input_file:

    # Create a CSV reader for a tab-delimited file with a header
    reader = csv.DictReader(input_file, delimiter='\t')

    # Iterate through records, each record is a dictionary
    for record in reader:
        key = record.get('code')
        id2name[key] = record.get('name')

In [ ]:
# LEAVE AS-IS, it should print "Germany"

print(id2name["DEU"])

# 2. Read flows data into two graphs

## 2.1. Read amount traded

In [ ]:
# Read the two graphs gOld and gNew

gOld = nx.DiGraph()
# Open a file for reading in text mode
with open(INPUT_TRADE_OLD, "rt") as input_file:

    # Create a CSV reader for a tab-delimited file with a header
    reader = csv.DictReader(input_file, delimiter='\t')

    # Iterate through records, each record is a dictionary
    for record in reader:
        from_node = record.get('from')
        to_node = record.get('to')
        amount_mill = round(float(record.get('amount'))/1000000)
        
        if from_node in id2name and to_node in id2name and amount_mill != 0:
            if from_node not in gOld:
                gOld.add_node(from_node)
            if to_node not in gOld:
                gOld.add_node(to_node)
            gOld.add_edge(from_node, to_node, weight=amount_mill)

gNew = nx.DiGraph()
# Open a file for reading in text mode
with open(INPUT_TRADE_NEW, "rt") as input_file:

    # Create a CSV reader for a tab-delimited file with a header
    reader = csv.DictReader(input_file, delimiter='\t')

    # Iterate through records, each record is a dictionary
    for record in reader:
        from_node = record.get('from')
        to_node = record.get('to')
        amount_mill = round(float(record.get('amount'))/1000000)
        
        if from_node in id2name and to_node in id2name:
            if from_node not in gNew:
                gNew.add_node(from_node)
            if to_node not in gNew:
                gNew.add_node(to_node)
            gNew.add_edge(from_node, to_node, weight=amount_mill)

In [ ]:
# LEAVE AS-IS

print("The {:d} graph contains {:d} nodes".format(YEAR_OLD, gOld.number_of_nodes()))
print("The {:d} graph contains {:d} nodes".format(YEAR_NEW, gNew.number_of_nodes()))

In [ ]:
# LEAVE AS-IS

for exporter in ["ESP", "PRT"]:
    for importer in ["FRA", "DEU"]:
        print("In {:d}, {:s} ({:s}) exported to {:s} ({:s}) goods and services for {:,d} USD millions".format(
            YEAR_OLD, exporter, id2name[exporter], importer, id2name[importer],
               gOld.get_edge_data(exporter, importer)["weight"]) )

        print("by {:d}, it exported {:,d} USD millions".format(YEAR_NEW, gNew.get_edge_data(exporter, importer)["weight"]))
        print("")
    

## 2.2. Compute totals

In [ ]:
# Compute totals into dictionaries exportsOld, exportsNew, importsOld, importsNew

def sum_weights(graph, weight_key='weight'):
    in_dict = dict([(element,0) for element in id2name])
    out_dict = dict([(element,0) for element in id2name])
    # u is the source, v the destination, w the weight
    for u, v, d in graph.edges(data=True):
        w = d[weight_key]
        out_dict[u] += w
        in_dict[v] += w
    return in_dict, out_dict

importsOld, exportsOld = sum_weights(gOld)
importsNew, exportsNew = sum_weights(gNew)

In [ ]:
# LEAVE AS-IS

for country in ['POL', 'ESP', 'CHL']:
    print("{:s} exported {:,d} USD Million in {:d} and {:,d} USD Million in {:d} (growth {:.1f}x),".format(
        id2name[country],
        exportsOld[country], YEAR_OLD,
        exportsNew[country], YEAR_NEW,
        exportsNew[country]/exportsOld[country]
    ))
    print(" and imported {:,d} USD Million in {:d} and {:,d} USD Million in {:d} (growth {:.1f}x).".format(
        importsOld[country], YEAR_OLD,
        importsNew[country], YEAR_NEW,
        importsNew[country]/importsOld[country]
    ))
    print()

In [ ]:
# LEAVE AS-IS

sorted_countries = sorted(id2name.keys())

flowsData = {
    'expOld': [(exportsOld[c] if c in exportsOld else 0) for c in sorted_countries],
    'impOld': [(importsOld[c] if c in importsOld else 0) for c in sorted_countries],
    'expNew': [(exportsNew[c] if c in exportsNew else 0) for c in sorted_countries], 
    'impNew': [(importsNew[c] if c in importsNew else 0) for c in sorted_countries],
}

flowsDF = pd.DataFrame(flowsData, index=sorted_countries)
flowsDF

# 3. Draw the exports graphs

In [ ]:
NODE_SIZE_MULTIPLIER = 0.01
EDGE_WIDTH_MULTIPLIER = 0.01

def plotGraph(graph, imp_exp, weight_key='weight'):
    plt.figure(figsize=(20,12))

    # Compute node sizes as a function of total exports
    node_sizes = []
    for node in graph.nodes():
        node_sizes.append(imp_exp[node] * NODE_SIZE_MULTIPLIER)

    # Compute edge widths as a function of exports
    edge_widths = []
    for u, v, d in graph.edges(data=True):
        weight = math.log(d[weight_key]) * EDGE_WIDTH_MULTIPLIER
        edge_widths.append(weight)
        
    # Determine node positions
    pos = nx.spring_layout(graph, iterations=100, weight="weight", k=2)

    # Draw nodes
    nx.draw_networkx_nodes(graph, pos, node_color="orange", node_size=node_sizes)

    # Draw edges
    nx.draw_networkx_edges(graph, pos, width=edge_widths)

    # Draw labels and display graph
    _ = nx.draw_networkx_labels(graph, pos, font_color="blue")

In [ ]:
plotGraph(gOld, exportsOld)

In [ ]:
plotGraph(gNew, exportsNew)

1. Similarities: the main countries that had the highest exports are still the same (e.g.: mainly North-American and Schenghen zone countries). Differences: now Finland (FIN), Portugal (PRT) receive much less exports (therefore has less imports) than before.
2. There are many countries that are close geographically and also close in these export graphs (e.g.: european countries or states in the Schengen zone), but the most significant ones are the United States of America (USA), Mexico (MEX) and Canada (CAN). 


# 4. Compute hubs and authorities

In [ ]:
def normalize(d):
    sum_values = sum(d.values())
    return dict([(key, d[key]/sum_values) for key in d])

In [ ]:
# LEAVE AS-IS

print(normalize({"a": 39, "b": 13, "c":26}))

In [ ]:
def hubs_authorities(graph, iter=100, weight_key='weight'):
    'Returns tuple of (hubs, authorities)'
    score_h = dict([entry, 1/len(graph.nodes())] for entry in graph.nodes()) # Initialized to 1/N
    score_a = dict([entry, 0] for entry in graph.nodes())

    for _ in range(iter):
        for u, v, w in graph.edges(data=True):
            score_a[v] += score_h[u] * w[weight_key]
        score_a = normalize(score_a)

        for u, v, w in graph.edges(data=True):
            score_h[u] += score_a[v] * w[weight_key]
        score_h = normalize(score_h)
    
    return (score_h, score_a)

In [ ]:
# LEAVE AS-IS
# Execution should be very fast (a few seconds maximum)

print("Computing for {:d}".format(YEAR_OLD))
(hOld,aOld) = hubs_authorities(gOld)

print("Computing for {:d}".format(YEAR_NEW))
(hNew,aNew) = hubs_authorities(gNew)

In [ ]:
# LEAVE AS-IS

flowsDF['hOld'] = pd.Series(hOld)
flowsDF['aOld'] = pd.Series(aOld)
flowsDF['hNew'] = pd.Series(hNew)
flowsDF['aNew'] = pd.Series(aNew)

flowsDF

In [ ]:
# Print top countries by exports and top countries by hub score

display(flowsDF.sort_values(by='expNew', ascending=False).head(10)[['expNew', 'hNew']])
display(flowsDF.sort_values(by='hNew', ascending=False).head(10)[['hNew', 'expNew']])

By construction of of the HITS algorithm (Hubs and Authorities), the hub scores share a very positive relation with the outdegree of a node, thus with the exports of a country in this case. That is mainly because the hub scores are computed by adding a weighed amount of the score of the nodes at which it is pointing. 

That is why we can see that even though we are sorting first by exports and then by hub scores we can see that the list is almost the same (besides BEL and ESP).

In [ ]:
# Print top countries by imports and top countries by authority score

display(flowsDF.sort_values(by='impNew', ascending=False).head(10)[['impNew', 'aNew']])
display(flowsDF.sort_values(by='aNew', ascending=False).head(10)[['aNew', 'impNew']])

Similarly as before, by construction of of the HITS algorithm (Hubs and Authorities), the authority scores share a very positive relation with the indegree of a node, thus with the imports of a country in this case. That is mainly because the authority scores are computed by adding a weighed amount of the score of the nodes that are pointing to it. 

That is why we can see that even though we are sorting first by imports and then by authority scores we can see that the list is almost the same.

# 5. Comparison of hub/export, authority/import scores

In [ ]:
def plot_scatter(a, b, x_label='x axis label', y_label='y axis label'):
    'a and b are dictionaries'
    # Create log-log plot
    plt.figure(figsize=(20,10))
    plt.loglog()
    plt.xlabel(x_label, {'size': '22'})
    plt.ylabel(y_label, {'size': '22'})

    # Add a diagonal line
    plt.plot([min(a.values()),max(a.values())], [min(b.values()),max(b.values())], '-.', lw=2)

    # Do the scatter plot with texts
    for country in set(a.keys()).intersection(set(b.keys())):
        plt.text(a[country], b[country], country, {'size': '12'})

In [ ]:
# LEAVE AS-IS: print plots for the newer dataset

plot_scatter( exportsNew, hNew, "Exports [Millions of USD]", "Hub score" )
plot_scatter( importsNew, aNew, "Imports [Millions of USD]", "Authority score" )

We can clearly see with the above plots that the exports are positively correlated with the hub score of a node, and the same with the imports and the authority scores. Mainly we can see that because as one increases, the other one does it, too.

# Extra section

In [ ]:
OUTPUT_GNEW = 'gNew-edges.csv'
OUTPUT_HUB_SCORES_NEW = 'hub-scores-new.csv'

with io.open(OUTPUT_GNEW, 'w') as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
    writer.writerow(['From', 'To', 'Weight'])
    for u, v, w in gNew.edges(data=True):
        writer.writerow([u, v, w['weight']])
        
with io.open(OUTPUT_HUB_SCORES_NEW, 'w') as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
    writer.writerow(['Node', 'hScore', 'aScore'])
    for key in flowsDF.index:
        writer.writerow([key, flowsDF.loc[key]['hNew'], flowsDF.loc[key]['aNew']])

In [ ]:
from IPython.display import Image

In [ ]:
Image(url='extra-network.png')

In [ ]:
Image(url='extra-legend.gif')

We can see that using affinity propagation clustering 5 clusters are generated: 
* The green cluster is mainly composed of states in the Schengen zone, which trade very frequently probably because of the ease of import/export due to international treaties in such zone.
* The purple cluster is composed of American states (e.g.: USA, Mexico or Canada) but also of states that trade frequently with them (e.g.: Ireland, Israel or Japan), probably because of geographical ease, treaties or political international agreements.
* The orange cluster is composed of New Zeland and Australia, in which I assume that they trade very frequently because of their geographical positions.
* The two last clusters (yellow for Iceland and blue for Estonia) only contain one state each, therefore probably those are states that do not trade very frequently with any other state, thus they are quite isolated.

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>